In [ ]:
# chat model is the model used


In [ ]:
from langchain_groq import ChatGroq
import os

# Set the environment variable directly (replace with your actual key)
os.environ["GROQ_API_KEY"] = "your api key"

# Now this will fetch the key successfully
api_key = os.environ.get("GROQ_API_KEY")

# Initialize the Groq LLM
llm = ChatGroq(
    groq_api_key=api_key,
    model_name="llama-3.3-70b-versatile",
    temperature = 0.7
)

In [ ]:
response = llm.invoke("suggest me english movie")
response

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
messages  = [
    SystemMessage(content = "translate the following text to spanish"),
    HumanMessage(content = "hello world")
]
llm.invoke(messages)

In [ ]:
response.content

In [ ]:
from langchain_core.prompts import PromptTemplate
movie_title_template  = PromptTemplate.from_template("suggest me title of of movie to watch in language {language} and of genre {genre}")
movie_title_prompt  = movie_title_template.invoke(
    {
        "language":"english", "genre":"horror"
    }
)

response = llm.invoke(movie_title_prompt)


In [ ]:
response.content

In [ ]:
from langchain_core.output_parsers import StrOutputParser
# chains every task is a chain and all are runnable
movie_title_chain = movie_title_template | llm  | StrOutputParser()
movie_title_chain.invoke({"language": "hindi", "genre":"horror"})

In [ ]:
movie_summary = PromptTemplate.from_template("write a short summary of theBhool Bhulaiyaa ")


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Redefine movie_summary to accept a dynamic movie title
movie_summary = PromptTemplate.from_template("write a short summary of {movie_title}")

# The chain needs to pass the original input ('language', 'genre') to movie_title_chain,
# then take the output of movie_title_chain (which is a string) and
# map it to the 'movie_title' key for the movie_summary prompt template.

composed_chain = (
    {
        "movie_title": movie_title_chain, # movie_title_chain is invoked with the input of composed_chain
        "original_input": RunnablePassthrough() # Keep original input if needed later, but for movie_summary it's just movie_title
    }
    | movie_summary # Now movie_summary receives {'movie_title': '...', 'original_input': {...}}
    | llm
    | StrOutputParser()
)

# Invoke the chain with the required input for movie_title_chain
result = composed_chain.invoke(
    {
        "language":"hindi", "genre":"horror"
    }
)
print(result)

In [ ]:
from langchain_core.runnables import RunnableLambda

print_title_step = RunnableLambda(lambda x: print(f"movie title: {movie_title_chain}"))

composed_chain = {"movie_title":movie_title_chain} | print_title_step | movie_summary | llm | StrOutputParser()

In [ ]:
composed_chain = {"movie_title": movie_title_chain} | RunnableLambda(lambda x: print(f"movie title: {movie_title_chain}")) | movie_summary | llm | StrOutputParser()

In [ ]:
# 1. SEQUENTIAL FLOW
from langchain_core.runnables import RunnableSequence

composed_chain = RunnableSequence({"movie_title": movie_title_chain},print_title_step, movie_summary, llm, StrOutputParser())

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

# Redefine movie_summary to accept a dynamic movie title
movie_summary = PromptTemplate.from_template("write a short summary of {movie_title}")

# The chain needs to pass the original input ('language', 'genre') to movie_title_chain,
# then take the output of movie_title_chain (which is a string) and
# map it to the 'movie_title' key for the movie_summary prompt template.

composed_chain = (
    {
        "movie_title": movie_title_chain, # movie_title_chain is invoked with the input of composed_chain
        "original_input": RunnablePassthrough() # Keep original input if needed later, but for movie_summary it's just movie_title
    }
    | movie_summary # Now movie_summary receives {'movie_title': '...', 'original_input': {...}}
    | llm
    | StrOutputParser()
)

# Invoke the chain with the required input for movie_title_chain
result = composed_chain.invoke(
    {
        "language":"hindi", "genre":"horror"
    }
)
print(result)

In [ ]:
# PARALLEL EXECUTION
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel

translate_hindi_chain  = ChatPromptTemplate.from_template("translate the summary {movie_summary} to hindi") | llm |  StrOutputParser()
translate_spanish_chain = ChatPromptTemplate.from_template("translate the summary {movie_summary} to spanish") |llm | StrOutputParser()

translate_runnable  = RunnableParallel(hindi_translate  = translate_hindi_chain, spanish_translate = translate_spanish_chain)

# Pass the 'result' (which contains the movie summary string) with the correct key 'movie_summary'
translate_runnable.invoke({"movie_summary": result})